## Setup

In [1]:
import pandas as pd
import numpy as np
import os
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter

SEED = 42
np.random.seed(SEED)

# ── Paths ──
IMAGES_DIR = "Images"                          # Raw images + metadata xlsx
METADATA_FILE = os.path.join(IMAGES_DIR, "release_midas.xlsx")
OUTPUT_CSV = "instances.csv"                   # Output manifest (saved in current directory)

# ── Quality thresholds ──
MIN_RESOLUTION = 64
MAX_ASPECT_RATIO = 4

# ── Cancer type classification ──
# Malignant
CANCER_MAP = {
    "Melanoma": ["melanoma"],
    "BCC": ["bcc"],
    "SCC": ["scc", "sccis"],
    "Actinic_Keratosis": ["ak"],
    # Benign (preserving subtypes)
    "Melanocytic_Nevus": ["melanocytic nevus"],
    "Seborrheic_Keratosis": ["seborrheic keratosis"],
    "Dermatofibroma": ["dermatofibroma"],
    "Hemangioma": ["hemangioma"],
    "Fibrous_Papule": ["fibrous papule"],
    # Ambiguous (will be excluded from binary classification)
    "Uncertain_Melanocytic": ["melanocytic lesion", "melanocytic tumor"],
    "Other_Inflammatory": ["non-neoplastic", "inflammatory", "infectious"],
}

# ── Binary label mapping (for training) ──
# Ambiguous categories (Uncertain_Melanocytic, Other_Inflammatory, Unclassified)
# are intentionally omitted — they are excluded from the dataset, consistent with
# the MRA-MIDAS baseline benchmarking paper (see EXCLUSION_ANALYSIS.md).
BINARY_MAP = {
    # Malignant (label = 1)
    "Melanoma": 1,
    "BCC": 1,
    "SCC": 1,
    "Actinic_Keratosis": 1,
    "Malignant_Other": 1,
    # Benign (label = 0)
    "Melanocytic_Nevus": 0,
    "Seborrheic_Keratosis": 0,
    "Dermatofibroma": 0,
    "Hemangioma": 0,
    "Fibrous_Papule": 0,
    "Other_Benign": 0,
}

print(f"Images dir:  {os.path.abspath(IMAGES_DIR)}")
print(f"Metadata:    {os.path.abspath(METADATA_FILE)}")
print(f"Output:      {os.path.abspath(OUTPUT_CSV)}")
print(f"Quality:     min {MIN_RESOLUTION}px, max aspect {MAX_ASPECT_RATIO}:1")

Images dir:  c:\Coding\SkinCancerImageClassificationResearch\DataCleaning\Images
Metadata:    c:\Coding\SkinCancerImageClassificationResearch\DataCleaning\Images\release_midas.xlsx
Output:      c:\Coding\SkinCancerImageClassificationResearch\DataCleaning\instances.csv
Quality:     min 64px, max aspect 4:1


## 1. Load & Filter Metadata

In [2]:
def classify_pathology(path_str):
    """Map a midas_path string to a cancer-type category (None if invalid)."""
    if pd.isna(path_str) or not isinstance(path_str, str):
        return None
    path_lower = path_str.lower().strip()
    for cancer_type, keywords in CANCER_MAP.items():
        if any(kw in path_lower for kw in keywords):
            return cancer_type
    # Catch-all for any remaining benign cases
    if "benign" in path_lower:
        return "Other_Benign"
    # Catch-all for malignant cases not captured above
    if "malignant" in path_lower:
        return "Malignant_Other"
    return "Unclassified"

# Load raw metadata
ds = pd.read_excel(METADATA_FILE)
ds["cancer_type"] = ds["midas_path"].apply(classify_pathology)

print(f"Total metadata rows: {len(ds)}")
print(f"  Invalid pathology (excluded): {ds['cancer_type'].isna().sum()}")

# Filter: valid pathology + non-virtual distance + dermoscope images only
dscope = ds[
    (ds["cancer_type"].notna()) &
    (ds["midas_distance"] == "dscope")
].copy()

print(f"  Virtual distance (excluded):  {(ds['midas_distance'] == 'n/a - virtual').sum()}")
print(f"\nDermoscope rows after filtering: {len(dscope)}")
print(f"\nCancer type distribution:")
print(dscope["cancer_type"].value_counts().to_string())

Total metadata rows: 3416
  Invalid pathology (excluded): 503
  Virtual distance (excluded):  260

Dermoscope rows after filtering: 965

Cancer type distribution:
cancer_type
BCC                      202
Melanocytic_Nevus        192
Other_Benign             138
SCC                      123
Seborrheic_Keratosis      79
Melanoma                  79
Actinic_Keratosis         64
Uncertain_Melanocytic     39
Dermatofibroma            16
Other_Inflammatory        13
Hemangioma                10
Fibrous_Papule             6
Malignant_Other            4


## 2. Quality-Check Images & Build Instance Manifest

In [3]:
def validate_image(path, min_res=MIN_RESOLUTION, max_ar=MAX_ASPECT_RATIO):
    """Check that an image is readable and meets quality thresholds."""
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            w, h = img.size
            if w < min_res or h < min_res:
                return False, f"too small ({w}x{h})"
            if max(w / h, h / w) > max_ar:
                return False, f"extreme aspect ratio ({max(w/h, h/w):.1f}:1)"
            return True, "ok"
    except Exception as e:
        return False, f"unreadable ({e})"

# Validate each dermoscope image and track results
valid_rows = []
counters = {"valid": 0, "missing": 0, "quality_fail": 0}
filter_reasons = []

for _, row in tqdm(dscope.iterrows(), total=len(dscope), desc="Validating images"):
    filepath = os.path.join(IMAGES_DIR, row["midas_file_name"])

    if not os.path.exists(filepath):
        counters["missing"] += 1
        continue

    ok, reason = validate_image(filepath)
    if not ok:
        counters["quality_fail"] += 1
        filter_reasons.append(reason)
        continue

    counters["valid"] += 1
    valid_rows.append(row)

valid_dscope = pd.DataFrame(valid_rows)

print(f"\nImage validation results:")
print(f"  Valid:          {counters['valid']}")
print(f"  Missing file:   {counters['missing']}")
print(f"  Quality fail:   {counters['quality_fail']}")
if filter_reasons:
    print(f"\n  Quality failure breakdown:")
    for reason, count in Counter(filter_reasons).most_common():
        print(f"    {reason}: {count}")

Validating images:   0%|          | 0/965 [00:00<?, ?it/s]


Image validation results:
  Valid:          965
  Missing file:   0
  Quality fail:   0


## 3. Group by Instance & Save Manifest

In [4]:
# Group validated dermoscope images by instance (record + pathology + location)
manifest_rows = []
instance_id = 0

for (record_id, path_val, location), group in valid_dscope.groupby(
    ["midas_record_id", "midas_path", "midas_location"]
):
    filenames = group["midas_file_name"].tolist()
    if not filenames:
        continue

    instance_id += 1
    manifest_rows.append({
        "instance_id": f"instance_{instance_id:04d}",
        "record_id": record_id,
        "cancer_type": group["cancer_type"].iloc[0],
        "pathology": path_val,
        "location": location,
        "dscope_files": ";".join(filenames),
        "n_dscope": len(filenames),
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df["binary_label"] = manifest_df["cancer_type"].map(BINARY_MAP)

# Exclude ambiguous instances (cancer_type not in BINARY_MAP → NaN binary_label)
ambiguous = manifest_df[manifest_df["binary_label"].isna()]
if len(ambiguous) > 0:
    print(f"Excluding {len(ambiguous)} ambiguous instances:")
    print(ambiguous["cancer_type"].value_counts().to_string())
    print()

manifest_df = manifest_df[manifest_df["binary_label"].notna()].copy()
manifest_df["binary_label"] = manifest_df["binary_label"].astype(int)

# Re-index instance IDs after exclusion
manifest_df["instance_id"] = [f"instance_{i+1:04d}" for i in range(len(manifest_df))]
manifest_df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {OUTPUT_CSV}")
print(f"\n{'='*60}")
print(f"DATASET SUMMARY")
print(f"{'='*60}")
print(f"Instances:              {len(manifest_df)}")
print(f"Total dermoscope imgs:  {manifest_df['n_dscope'].sum()}")
print(f"Avg imgs/instance:      {manifest_df['n_dscope'].mean():.2f}")

print(f"\nBy cancer type:")
print(manifest_df["cancer_type"].value_counts().sort_index().to_string())

print(f"\nBinary distribution:")
for label in [0, 1]:
    count = (manifest_df["binary_label"] == label).sum()
    pct = 100 * count / len(manifest_df)
    class_name = "Benign" if label == 0 else "Malignant"
    print(f"  {class_name} ({label}): {count} ({pct:.1f}%)")

print(f"\nDermoscope images per instance:")
print(manifest_df["n_dscope"].value_counts().sort_index().to_string())

print(f"\nPreview:")
display(manifest_df.head(10))

Excluding 51 ambiguous instances:
cancer_type
Uncertain_Melanocytic    38
Other_Inflammatory       13

Saved instances.csv

DATASET SUMMARY
Instances:              906
Total dermoscope imgs:  913
Avg imgs/instance:      1.01

By cancer type:
cancer_type
Actinic_Keratosis        63
BCC                     200
Dermatofibroma           16
Fibrous_Papule            6
Hemangioma               10
Malignant_Other           4
Melanocytic_Nevus       190
Melanoma                 77
Other_Benign            138
SCC                     123
Seborrheic_Keratosis     79

Binary distribution:
  Benign (0): 439 (48.5%)
  Malignant (1): 467 (51.5%)

Dermoscope images per instance:
n_dscope
1    899
2      7

Preview:


,instance_id,record_id,cancer_type,pathology,location,dscope_files,n_dscope,binary_label
0,instance_0001,1,Melanocytic_Nevus,benign-melanocytic nevus,l lower back,s-prd-398967587.jpg,1,0
1,instance_0002,1,SCC,malignant- sccis,chest,s-prd-398966845.jpg,1,1
2,instance_0003,2,Other_Benign,benign-other,right upper eyelid,s-prd-399002046.jpg,1,0
3,instance_0004,4,Melanocytic_Nevus,benign-melanocytic nevus,left upper back,s-prd-399396846.jpg,1,0
4,instance_0005,5,Seborrheic_Keratosis,benign-seborrheic keratosis,r forehead,s-prd-450517756.jpg,1,0
5,instance_0006,5,Actinic_Keratosis,malignant- ak,nasal bridge,s-prd-401873031.jpg,1,1
6,instance_0007,5,BCC,malignant- bcc,r posterior shoulder,s-prd-450517936.jpg,1,1
7,instance_0008,5,SCC,malignant- scc,r dorsal hand,s-prd-401872340.jpg,1,1
8,instance_0009,6,SCC,malignant- scc,r flank,s-prd-401900819.jpg,1,1
9,instance_0010,8,BCC,malignant- bcc,l cheek,s-prd-401919763.jpg,1,1
